# Python Package Management with pip and uv

## Introduction to Package Management
---

You already know how to create **virtual environments** and configure your applications with **environment variables**.

Now you'll learn how to efficiently **install and manage packages** using `pip` and the modern `uv` tool.

1. [pip Basics](#pip-Basics),
    - [Installing Packages](#Installing-Packages),
    - [Upgrading and Uninstalling](#Upgrading-and-Uninstalling),
    - [Listing Installed Packages](#Listing-Installed-Packages),
2. [requirements.txt](#requirements.txt),
    - [Creating requirements.txt](#Creating-requirements.txt),
    - [Installing from requirements.txt](#Installing-from-requirements.txt),
    - [Best Practices](#Best-Practices),
3. [Dependency Versioning](#Dependency-Versioning),
    - [Version Specifiers](#Version-Specifiers),
    - [Pinning vs Ranges](#Pinning-vs-Ranges),
4. [uv - The Modern Alternative](#uv---The-Modern-Alternative),
    - [Why uv](#Why-uv),
    - [Installing uv](#Installing-uv),
    - [Using uv](#Using-uv),
5. [🧠 EXERCISE 🧠, Manage Project Dependencies](#🧠-EXERCISE-🧠,-Manage-Project-Dependencies).

<br>

## pip Basics

---

Think of `pip` as your **shopping assistant** for Python.

Instead of going to a store, you go to **PyPI** (Python Package Index) - a huge warehouse with over 500,000 packages.

<br>

`pip` helps you:
- **Install** packages (buy items)
- **Upgrade** packages (get newer versions)
- **Uninstall** packages (return items)
- **List** what you have (check your inventory)

<br>

### Installing Packages

---

The most common pip command is `pip install`:

<br>

**Linux/macOS:**
```bash
# Install a single package
pip install requests

# Install multiple packages
pip install requests pandas numpy

# Install a specific version
pip install requests==2.31.0

# Install minimum version
pip install requests>=2.25.0
```

**Windows:**
```powershell
# Same commands work on Windows
pip install requests
pip install requests pandas numpy
pip install requests==2.31.0
```

<br>

**Important**: Always install packages in a **virtual environment**, not globally!

```bash
# BAD: Installing globally
pip install requests  # Don't do this!

# EXCELLENT: Installing in virtual environment
source .venv/bin/activate  # Activate first!
pip install requests       # Now it's isolated
```

<br>

Let's see what happens when you install a package:

In [ ]:
# After installing requests, you can import it
import requests

# Check the installed version
print(f"requests version: {requests.__version__}")

# Make a simple HTTP request
response = requests.get("https://httpbin.org/get")
print(f"Status code: {response.status_code}")

<br>

### Upgrading and Uninstalling

---

**Upgrade a package to the latest version:**

```bash
pip install --upgrade requests
# or shorter:
pip install -U requests
```

<br>

**Upgrade pip itself:**

```bash
pip install --upgrade pip
```

<br>

**Uninstall a package:**

```bash
pip uninstall requests

# Skip confirmation prompt
pip uninstall -y requests
```

<br>

### Listing Installed Packages

---

**List all installed packages:**

```bash
pip list
```

**Example output:**
```
Package    Version
---------- -------
certifi    2024.2.2
charset-normalizer 3.3.2
idna       3.6
pip        24.0
requests   2.31.0
urllib3    2.2.1
```

<br>

**Show details about a specific package:**

```bash
pip show requests
```

**Example output:**
```
Name: requests
Version: 2.31.0
Summary: Python HTTP for Humans.
Home-page: https://requests.readthedocs.io
Author: Kenneth Reitz
Requires: certifi, charset-normalizer, idna, urllib3
Required-by:
```

<br>

**Check for outdated packages:**

```bash
pip list --outdated
```

**Example output:**
```
Package    Version Latest Type
---------- ------- ------ -----
requests   2.28.0  2.31.0 wheel
urllib3    1.26.0  2.2.1  wheel
```

<br>

## requirements.txt

---

A `requirements.txt` file is a **shopping list** for your project.

It lists all the packages your project needs, so anyone can install them with a single command.

<br>

### Creating requirements.txt

---

**Method 1: pip freeze (captures current state)**

```bash
pip freeze > requirements.txt
```

This creates a file with **all installed packages and their exact versions**:

```
certifi==2024.2.2
charset-normalizer==3.3.2
idna==3.6
requests==2.31.0
urllib3==2.2.1
```

<br>

**Method 2: Manual creation (recommended for new projects)**

Create the file manually with only your **direct dependencies**:

```
# requirements.txt
requests==2.31.0
pandas>=2.0.0
python-dotenv~=1.0.0
```

This is cleaner because it doesn't include sub-dependencies.

<br>

### Installing from requirements.txt

---

To install all packages from a requirements file:

```bash
pip install -r requirements.txt
```

<br>

This is what your colleagues (or CI/CD) will run after cloning your project:

```bash
# Typical setup workflow
git clone https://github.com/user/project.git
cd project
python -m venv .venv
source .venv/bin/activate  # or .venv\Scripts\activate on Windows
pip install -r requirements.txt
```

<br>

### Best Practices

---

**1. Use comments to organize your requirements:**

```
# requirements.txt

# Web framework
flask==3.0.0
flask-cors==4.0.0

# Database
sqlalchemy==2.0.25
psycopg2-binary==2.9.9

# Utilities
python-dotenv==1.0.0
requests==2.31.0
```

<br>

**2. Separate development dependencies:**

```
# requirements.txt - Production dependencies
flask==3.0.0
sqlalchemy==2.0.25

# requirements-dev.txt - Development dependencies
-r requirements.txt  # Include production deps
pytest==8.0.0
black==24.1.0
mypy==1.8.0
```

```bash
# Install for production
pip install -r requirements.txt

# Install for development (includes production + dev tools)
pip install -r requirements-dev.txt
```

<br>

**3. Keep requirements.txt in version control:**

```
my_project/
├── .venv/                  # NOT in git
├── .gitignore
├── requirements.txt        # IN git
├── requirements-dev.txt    # IN git
└── src/
```

<br>

## Dependency Versioning

---

### Version Specifiers

---

Python packages use **Semantic Versioning** (SemVer): `MAJOR.MINOR.PATCH`

| Part | When to Increment | Example |
|------|-------------------|----------|
| **MAJOR** | Breaking changes | 1.0.0 → 2.0.0 |
| **MINOR** | New features (backward compatible) | 1.0.0 → 1.1.0 |
| **PATCH** | Bug fixes | 1.0.0 → 1.0.1 |

<br>

**Version specifiers in requirements.txt:**

| Specifier | Meaning | Example |
|-----------|---------|----------|
| `==` | Exact version | `requests==2.31.0` |
| `>=` | Minimum version | `requests>=2.25.0` |
| `<=` | Maximum version | `requests<=2.31.0` |
| `>`, `<` | Exclusive bounds | `requests>2.0.0` |
| `!=` | Exclude version | `requests!=2.30.0` |
| `~=` | Compatible release | `requests~=2.31.0` |
| `>=,<` | Range | `requests>=2.25.0,<3.0.0` |

<br>

**The `~=` (compatible release) operator:**

```
requests~=2.31.0
# Equivalent to: requests>=2.31.0,<2.32.0
# Allows: 2.31.0, 2.31.1, 2.31.2, ...
# Blocks: 2.32.0, 3.0.0

requests~=2.31
# Equivalent to: requests>=2.31,<3.0
# Allows: 2.31.0, 2.32.0, 2.99.0, ...
# Blocks: 3.0.0
```

<br>

### Pinning vs Ranges

---

**Pinning** = exact versions (`==`)

**Ranges** = flexible versions (`>=`, `~=`, etc.)

<br>

| Approach | Pros | Cons |
|----------|------|------|
| **Pinning** | Reproducible, no surprises | Manual updates, security patches delayed |
| **Ranges** | Auto-updates, security patches | Less reproducible, possible breakage |

<br>

**Recommendation:**

```
# For applications (deployed code): Pin exact versions
flask==3.0.0
requests==2.31.0
sqlalchemy==2.0.25

# For libraries (packages others install): Use compatible ranges
requests>=2.25.0,<3.0.0
sqlalchemy~=2.0
```

<br>

## uv - The Modern Alternative

---

### Why uv

---

`uv` is a **blazingly fast** Python package manager written in Rust.

It's designed as a **drop-in replacement for pip** that's 10-100x faster.

<br>

| Feature | pip | uv |
|---------|-----|----|
| **Speed** | Slow | 10-100x faster |
| **Caching** | Basic | Advanced |
| **Parallel downloads** | No | Yes |
| **Lock files** | No | Yes (uv.lock) |
| **Virtual env creation** | Separate tool | Built-in |

<br>

**Speed comparison example:**

```bash
# Installing Django with pip: ~15 seconds
pip install django

# Installing Django with uv: ~0.5 seconds
uv pip install django
```

<br>

### Installing uv

---

**Linux/macOS:**
```bash
# Using curl (recommended)
curl -LsSf https://astral.sh/uv/install.sh | sh

# Or using pip
pip install uv
```

**Windows (PowerShell):**
```powershell
# Using PowerShell
powershell -c "irm https://astral.sh/uv/install.ps1 | iex"

# Or using pip
pip install uv
```

<br>

**Verify installation:**

```bash
uv --version
```

<br>

### Using uv

---

`uv` commands are almost identical to pip - just prefix with `uv`:

<br>

**Package installation:**

```bash
# Install packages
uv pip install requests pandas

# Install from requirements.txt
uv pip install -r requirements.txt

# Upgrade a package
uv pip install --upgrade requests

# Uninstall
uv pip uninstall requests
```

<br>

**Virtual environment management (built-in!):**

```bash
# Create virtual environment
uv venv .venv

# Create with specific Python version
uv venv .venv --python 3.11

# Activate (same as before)
source .venv/bin/activate  # Linux/macOS
.venv\Scripts\activate     # Windows
```

<br>

**Listing and showing packages:**

```bash
# List installed packages
uv pip list

# Show package info
uv pip show requests

# Freeze dependencies
uv pip freeze > requirements.txt
```

<br>

**Complete workflow with uv:**

```bash
# Create project
mkdir my_project && cd my_project

# Create virtual environment with uv
uv venv .venv

# Activate
source .venv/bin/activate  # Linux/macOS
# .venv\Scripts\activate   # Windows

# Install packages (super fast!)
uv pip install requests pandas python-dotenv

# Save dependencies
uv pip freeze > requirements.txt
```

<br>

**Comparison cheat sheet:**

| Task | pip | uv |
|------|-----|----|
| Create venv | `python -m venv .venv` | `uv venv .venv` |
| Install package | `pip install requests` | `uv pip install requests` |
| Install from file | `pip install -r requirements.txt` | `uv pip install -r requirements.txt` |
| Upgrade package | `pip install -U requests` | `uv pip install -U requests` |
| Uninstall | `pip uninstall requests` | `uv pip uninstall requests` |
| List packages | `pip list` | `uv pip list` |
| Freeze | `pip freeze` | `uv pip freeze` |

<br>

## Summary

---

| Concept | Key Points |
|---------|------------|
| **pip** | Python's default package manager, `pip install`, `pip freeze` |
| **requirements.txt** | List of dependencies, one per line, version specifiers |
| **Version specifiers** | `==` (exact), `>=` (minimum), `~=` (compatible) |
| **Pinning** | Exact versions for apps, ranges for libraries |
| **uv** | Modern, fast alternative to pip, drop-in replacement |
| **Best practice** | Always use virtual environments, separate dev dependencies |

<br>

### 🧠 EXERCISE 🧠, Manage Project Dependencies

---

Practice the complete workflow for managing project dependencies:

1. Create a new project directory called `dependency_practice`
2. Create a virtual environment (using either `venv` or `uv`)
3. Activate the virtual environment
4. Install these packages: `requests`, `python-dotenv`, `rich`
5. Create a `requirements.txt` file with exact versions
6. Check which packages are installed and their versions
7. Uninstall `rich`
8. Verify it's removed from the installed packages
9. Update `requirements.txt` to reflect the change

<details>
    <summary>▶️ Solution (using pip - Linux/macOS)</summary>
    
```bash
# 1. Create project directory
mkdir dependency_practice
cd dependency_practice

# 2. Create virtual environment
python3 -m venv .venv

# 3. Activate
source .venv/bin/activate

# 4. Install packages
pip install requests python-dotenv rich

# 5. Create requirements.txt
pip freeze > requirements.txt
cat requirements.txt

# 6. Check installed packages
pip list

# 7. Uninstall rich
pip uninstall -y rich

# 8. Verify removal
pip list | grep -i rich  # Should show nothing

# 9. Update requirements.txt
pip freeze > requirements.txt
cat requirements.txt  # rich should be gone
```
</details>

<details>
    <summary>▶️ Solution (using pip - Windows PowerShell)</summary>
    
```powershell
# 1. Create project directory
mkdir dependency_practice
cd dependency_practice

# 2. Create virtual environment
python -m venv .venv

# 3. Activate
.venv\Scripts\Activate.ps1

# 4. Install packages
pip install requests python-dotenv rich

# 5. Create requirements.txt
pip freeze > requirements.txt
Get-Content requirements.txt

# 6. Check installed packages
pip list

# 7. Uninstall rich
pip uninstall -y rich

# 8. Verify removal
pip list | Select-String rich  # Should show nothing

# 9. Update requirements.txt
pip freeze > requirements.txt
Get-Content requirements.txt  # rich should be gone
```
</details>

<details>
    <summary>▶️ Solution (using uv - Linux/macOS)</summary>
    
```bash
# 1. Create project directory
mkdir dependency_practice
cd dependency_practice

# 2. Create virtual environment with uv (faster!)
uv venv .venv

# 3. Activate
source .venv/bin/activate

# 4. Install packages (much faster!)
uv pip install requests python-dotenv rich

# 5. Create requirements.txt
uv pip freeze > requirements.txt
cat requirements.txt

# 6. Check installed packages
uv pip list

# 7. Uninstall rich
uv pip uninstall rich

# 8. Verify removal
uv pip list | grep -i rich  # Should show nothing

# 9. Update requirements.txt
uv pip freeze > requirements.txt
cat requirements.txt  # rich should be gone
```
</details>

<br>

### 🧠 EXERCISE 🧠, Create a Proper Requirements Structure

---

Create a proper dependency structure for a web project:

1. Create `requirements.txt` with production dependencies:
   - `flask` (pinned to exact version)
   - `requests` (pinned to exact version)
   - `python-dotenv` (compatible release)

2. Create `requirements-dev.txt` with development dependencies:
   - Include production requirements
   - Add `pytest` (pinned)
   - Add `black` (pinned)

3. Add comments to organize the files

<details>
    <summary>▶️ Solution</summary>

**requirements.txt:**
```
# requirements.txt - Production dependencies
#
# Install with: pip install -r requirements.txt

# Web framework
flask==3.0.2

# HTTP client
requests==2.31.0

# Configuration
python-dotenv~=1.0.0
```

**requirements-dev.txt:**
```
# requirements-dev.txt - Development dependencies
#
# Install with: pip install -r requirements-dev.txt

# Include all production dependencies
-r requirements.txt

# Testing
pytest==8.0.2
pytest-cov==4.1.0

# Code formatting
black==24.2.0

# Linting
ruff==0.2.2
```

**Installation:**
```bash
# For production
pip install -r requirements.txt

# For development
pip install -r requirements-dev.txt
```
</details>

---